# Cluster Analysis: Poetic Features

In [2]:
import warnings

warnings.filterwarnings("ignore")

from mqdq import babble, elegy
from mqdq import line_analyzer as la

import numpy as np
import pandas as pd


from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from typing import Callable, Collection, Union

import glob
import os

from pathlib import Path

In [3]:
from pathlib import Path


def find_metrical_root():
    current = Path.cwd().resolve()

    for folder in [current, *current.parents]:
        if (folder / "corpus").is_dir():
            return folder

    raise FileNotFoundError("Metrical analysis root not found.")


BASE_DIR = find_metrical_root()
CORPUS_DIR = BASE_DIR / "corpus"

excluded_files = {
    "X3_machabeis.xml",
    "X4_certamina.xml",
}

files = sorted([
    fn for fn in CORPUS_DIR.glob("*.xml")
    if fn.name not in excluded_files
])

collection = []

for fn in files:
    parts = fn.stem.split("_")

    author = parts[0]
    title = parts[1]

    print("Sto leggendo:", fn.name)

    b = babble.Babbler.from_file(fn, name=title)
    b.author = author

    collection.append(b)

Sto leggendo: AI_anticlaudianus.xml
Lines: 4438 Corrupt: 43 Total Dropped: 43
Sto leggendo: AeC_medicaminum.xml
Lines: 4588 Corrupt: 174 Total Dropped: 174
Sto leggendo: AeC_viaticus.xml
Lines: 2373 Corrupt: 61 Total Dropped: 61
Sto leggendo: AeP_karolinus.xml
Lines: 2598 Corrupt: 89 Total Dropped: 89
Sto leggendo: BC_contemptu.xml
Lines: 2966 Corrupt: 62 Total Dropped: 62
Sto leggendo: B_judas.xml
Lines: 912 Corrupt: 55 Total Dropped: 55
Sto leggendo: DC_mathildis.xml
Lines: 2938 Corrupt: 142 Total Dropped: 142
Sto leggendo: E_biblia.xml
Lines: 1464 Corrupt: 56 Total Dropped: 56
Sto leggendo: FB_nuptis.xml
Lines: 4738 Corrupt: 103 Total Dropped: 103
Sto leggendo: GAp_roberti.xml
Lines: 2833 Corrupt: 59 Total Dropped: 59
Sto leggendo: GAr_philippis.xml
Lines: 1403 Corrupt: 52 Total Dropped: 52
Sto leggendo: GC_alexandreis.xml
Lines: 5511 Corrupt: 140 Total Dropped: 140
Sto leggendo: GM_merlini.xml
Lines: 1529 Corrupt: 37 Total Dropped: 37
Sto leggendo: GP_ligurinus.xml
Lines: 6575 Corr

In [4]:
for b in collection:
    print(f"Author: {b.author}, Name: {b.name}")

Author: AI, Name: anticlaudianus
Author: AeC, Name: medicaminum
Author: AeC, Name: viaticus
Author: AeP, Name: karolinus
Author: BC, Name: contemptu
Author: B, Name: judas
Author: DC, Name: mathildis
Author: E, Name: biblia
Author: FB, Name: nuptis
Author: GAp, Name: roberti
Author: GAr, Name: philippis
Author: GC, Name: alexandreis
Author: GM, Name: merlini
Author: GP, Name: ligurinus
Author: GVS, Name: poetria
Author: GiP, Name: historia
Author: HA, Name: planctus
Author: HP, Name: maiolichinus
Author: H, Name: epigrammata
Author: H, Name: maria
Author: IH, Name: architrenius
Author: II, Name: ylias
Author: M, Name: capitulorum
Author: M, Name: maurilii
Author: M, Name: victoris
Author: NLC, Name: passio
Author: OM, Name: ernestus
Author: OSE, Name: doctrina
Author: RC, Name: tancredi
Author: RT, Name: mauri
Author: SG, Name: ecclesiasten
Author: U10, Name: karolellus
Author: U11, Name: katherinae
Author: U1, Name: canticamariae
Author: U2, Name: liberesdre
Author: U3, Name: generati

In [5]:
sz = 150 
collection_chunked = []  

for b in collection:
    ll = b.raw_source
    chunks = [ll[i : i + sz] for i in range(0, len(ll), sz)]

    for i, chunk in enumerate(chunks):
        if len(chunk) == sz:  
            new_babbler = babble.Babbler(
                chunk, name=f"{b.name} - {i+1}"
            )  
            new_babbler.author = b.author  
            collection_chunked.append(new_babbler)

print([f"{x.author} - {x.name} - {len(x)}" for x in collection_chunked])

['AI - anticlaudianus - 1 - 150', 'AI - anticlaudianus - 2 - 150', 'AI - anticlaudianus - 3 - 150', 'AI - anticlaudianus - 4 - 150', 'AI - anticlaudianus - 5 - 150', 'AI - anticlaudianus - 6 - 150', 'AI - anticlaudianus - 7 - 150', 'AI - anticlaudianus - 8 - 150', 'AI - anticlaudianus - 9 - 150', 'AI - anticlaudianus - 10 - 150', 'AI - anticlaudianus - 11 - 150', 'AI - anticlaudianus - 12 - 150', 'AI - anticlaudianus - 13 - 150', 'AI - anticlaudianus - 14 - 150', 'AI - anticlaudianus - 15 - 150', 'AI - anticlaudianus - 16 - 150', 'AI - anticlaudianus - 17 - 150', 'AI - anticlaudianus - 18 - 150', 'AI - anticlaudianus - 19 - 150', 'AI - anticlaudianus - 20 - 150', 'AI - anticlaudianus - 21 - 150', 'AI - anticlaudianus - 22 - 150', 'AI - anticlaudianus - 23 - 150', 'AI - anticlaudianus - 24 - 150', 'AI - anticlaudianus - 25 - 150', 'AI - anticlaudianus - 26 - 150', 'AI - anticlaudianus - 27 - 150', 'AI - anticlaudianus - 28 - 150', 'AI - anticlaudianus - 29 - 150', 'AeC - medicaminum - 1

In [6]:
vectors = []

for b in collection_chunked:
    v = la.chunked_features(
        b.raw_source,
        n=len(b.raw_source)
    )

    v.insert(0, "Poem", b.name)
    v.insert(0, "Work", b.name.rsplit(" - ", 1)[0])
    v.insert(0, "Author", b.author)

    vectors.append(v)

collection_vect = pd.concat(vectors, ignore_index=True)

collection_vect.head()

,Author,Work,Poem,1SP,2SP,3SP,4SP,1CF,2CF,3CF,...,4DI,1SC,2SC,3SC,4SC,1WC,2WC,3WC,4WC,ELC
0,AI,anticlaudianus,anticlaudianus - 1,0.380000,0.513333,0.620000,0.666667,0.280000,0.746667,0.820000,...,0.446667,0.360000,0.573333,0.840000,0.613333,0.166667,0.153333,0.160000,0.080000,0.000000
1,AI,anticlaudianus,anticlaudianus - 2,0.386667,0.586667,0.653333,0.813333,0.306667,0.700000,0.826667,...,0.493333,0.340000,0.540000,0.840000,0.633333,0.140000,0.193333,0.146667,0.020000,0.000000
2,AI,anticlaudianus,anticlaudianus - 3,0.293333,0.573333,0.520000,0.746667,0.260000,0.713333,0.886667,...,0.500000,0.326667,0.620000,0.880000,0.633333,0.180000,0.160000,0.100000,0.026667,0.000000
3,AI,anticlaudianus,anticlaudianus - 4,0.440000,0.480000,0.606667,0.820000,0.260000,0.700000,0.886667,...,0.433333,0.393333,0.600000,0.886667,0.660000,0.120000,0.153333,0.106667,0.046667,0.006667
4,AI,anticlaudianus,anticlaudianus - 5,0.353333,0.586667,0.593333,0.780000,0.226667,0.680000,0.846667,...,0.566667,0.380000,0.526667,0.846667,0.580000,0.173333,0.200000,0.133333,0.053333,0.000000


In [7]:
collection_vect.to_csv("output.csv", index=False, encoding="utf-8")